In [ ]:
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from collections import deque
from PIL import Image

RNG = np.random.default_rng(42)

# Hyperparameters
N_HUBS      = 7        # number of hubs
N_DATA      = 70       # number of data points
LAM         = 0.2      # HP regulariser weight λ
ETA_0       = 0.012    # initial step size
LR_DECAY    = 0.012    # η_t = ETA_0 / (1 + t * LR_DECAY)
CW          = 0.3      # edge cost weight
REPULSION   = 0.12     # repulsion strength
MIN_SEP     = 0.13     # minimum hub separation
GRAD_CLIP   = 4.0      # gradient clipping threshold
EDGE_EVERY  = 20       # prune edges every N steps
N_ITER      = 300      # total GD iterations

GIF_FRAMES  = 50
GIF_FPS     = 10

CAND_THRESH = 0.6      # max distance for a candidate edge

BG_COLOR    = "white"
FG_COLOR    = "#111"


def make_scene():
    # data: N_HUBS clusters scattered across the unit square
    centers = RNG.uniform(0.12, 0.88, (N_HUBS, 2))
    assign  = RNG.integers(0, N_HUBS, N_DATA)
    data    = centers[assign] + RNG.normal(0, 0.07, (N_DATA, 2))
    data    = np.clip(data, 0.02, 0.98)

    # hubs: start on a jittered grid so they're spread across the space,
    # not biased toward where the data happens to fall
    cols = int(np.ceil(np.sqrt(N_HUBS)))
    rows = int(np.ceil(N_HUBS / cols))
    xs   = np.linspace(0.15, 0.85, cols)
    ys   = np.linspace(0.15, 0.85, rows)
    grid = np.array([[x, y] for y in ys for x in xs])[:N_HUBS]
    hubs = grid + RNG.normal(0, 0.05, grid.shape)
    hubs = np.clip(hubs, 0.05, 0.95)
    return data, hubs

# graph utilities
def prim_mst(hubs):
    n = len(hubs)
    in_tree, edges = {0}, []
    while len(in_tree) < n:
        best, bd = None, np.inf
        for i in in_tree:
            for j in range(n):
                if j not in in_tree:
                    d = np.linalg.norm(hubs[i] - hubs[j])
                    if d < bd:
                        bd, best = d, (i, j)
        if best is None: break
        in_tree.add(best[1]); edges.append(best)
    return edges

def canon(e):
    return (min(e), max(e))

def build_candidate_edges(hubs):
    n, edges = len(hubs), []
    for i in range(n):
        for j in range(i+1, n):
            if np.linalg.norm(hubs[i]-hubs[j]) < CAND_THRESH:
                edges.append(canon((i,j)))
    for e in prim_mst(hubs):
        c = canon(e)
        if c not in edges:
            edges.append(c)
    return edges

def is_connected_without(active, skip_idx, n):
    adj = [[] for _ in range(n)]
    for k,(i,j) in enumerate(active):
        if k != skip_idx:
            adj[i].append(j); adj[j].append(i)
    vis, q = {0}, deque([0])
    while q:
        cur = q.popleft()
        for nb in adj[cur]:
            if nb not in vis: vis.add(nb); q.append(nb)
    return len(vis) == n

def get_assignments(data, hubs):
    dists = np.linalg.norm(data[:,None] - hubs[None], axis=2)
    return np.argmin(dists, axis=1)

def bfs_path(active, n, src, dst):
    adj = [[] for _ in range(n)]
    for i,j in active: adj[i].append(j); adj[j].append(i)
    prev, vis, q = [-1]*n, [False]*n, deque([src])
    vis[src] = True
    while q:
        cur = q.popleft()
        if cur == dst: break
        for nb in adj[cur]:
            if not vis[nb]: vis[nb]=True; prev[nb]=cur; q.append(nb)
    path, cur = [], dst
    while cur != -1: path.insert(0, cur); cur = prev[cur]
    return path if (path and path[0]==src) else []

# gradient
def compute_gradients(hubs, data, active, asn):
    n     = len(hubs)
    grads = np.zeros_like(hubs)
    wi    = np.array([np.sum(asn==i) for i in range(n)], float)
    N2    = max(1, len(data)**2)

    # Fidelity
    for k in range(n):
        pts = data[asn==k]
        if not len(pts): continue
        diff = hubs[k] - pts
        d    = np.linalg.norm(diff, axis=1, keepdims=True).clip(1e-9)
        grads[k] += (diff/d).sum(axis=0)

    # Repulsion - smooth, not noisy
    for a in range(n):
        for b in range(a+1, n):
            diff = hubs[a] - hubs[b]
            d    = np.linalg.norm(diff).clip(1e-9)
            if d < 3*MIN_SEP:
                # smooth quadratic barrier instead of inverse-cube
                f = REPULSION * (3*MIN_SEP - d) / (d + 1e-6)
                direction = diff / d
                grads[a] -= f * direction
                grads[b] += f * direction

    # Complexity  1/4 w(i)w(j) path-length gradient
    for i in range(n):
        for j in range(i+1, n):
            path = bfs_path(active, n, i, j)
            if len(path) < 2: continue
            w = 0.25 * wi[i] * wi[j] / N2
            for idx, k in enumerate(path):
                nbrs  = []
                if idx > 0:           nbrs.append(path[idx-1])
                if idx < len(path)-1: nbrs.append(path[idx+1])
                for l in nbrs:
                    diff = hubs[k]-hubs[l]
                    d    = np.linalg.norm(diff).clip(1e-9)
                    grads[k] += w * 2 * diff / d

    # HP regulariser
    for i,j in active:
        diff = hubs[i]-hubs[j]
        grads[i] += (2*LAM/n) * diff
        grads[j] -= (2*LAM/n) * diff

    # Cost
    nE = max(1, len(active))
    for i,j in active:
        diff = hubs[i]-hubs[j]
        d    = np.linalg.norm(diff).clip(1e-9)
        grads[i] += (2*CW/nE) * diff/d
        grads[j] -= (2*CW/nE) * diff/d

    return grads

def compute_loss(hubs, data, active, asn):
    n  = len(hubs)
    wi = np.array([np.sum(asn==i) for i in range(n)], float)
    F  = 0.0
    for k in range(n):
        pts = data[asn==k]
        if len(pts): F += np.linalg.norm(hubs[k]-pts, axis=1).sum()
    for i in range(n):
        for j in range(i+1, n):
            path = bfs_path(active, n, i, j)
            if len(path) < 2: continue
            L = sum(np.linalg.norm(hubs[path[t]]-hubs[path[t+1]])
                    for t in range(len(path)-1))
            F += 0.25*wi[i]*wi[j]*L
    for i,j in active:
        diff = hubs[i]-hubs[j]
        F += LAM*np.dot(diff,diff) + CW*np.linalg.norm(diff)
    return F

def edge_score(hubs, active, asn, i, j):
    n    = len(hubs)
    wi   = np.sum(asn==i); wj = np.sum(asn==j)
    nE   = max(1, len(active)); N2 = max(1, len(asn)**2)
    dist = np.linalg.norm(hubs[i]-hubs[j])
    return (LAM/n)*dist**2 + (CW/nE)*dist + 0.25*wi*wj*dist/N2

def prune_edges(hubs, active, asn):
    n, active = len(hubs), list(active)
    max_rm    = max(1, len(active)//4)
    for _ in range(max_rm):
        if len(active) <= n-1: break
        scored  = sorted(enumerate(active),
                         key=lambda x: -edge_score(hubs, active, asn, *x[1]))
        removed = False
        for idx,(i,j) in scored:
            if is_connected_without(active, idx, n):
                active.pop(idx); removed = True; break
        if not removed: break
    # guarantee MST connectivity
    for e in prim_mst(hubs):
        c = canon(e)
        if c not in [canon(f) for f in active]:
            active.append(c)
    return active

# Optimization
def run_optimization():
    data, hubs = make_scene()
    active = [canon(e) for e in prim_mst(hubs)]
    asn    = get_assignments(data, hubs)
    losses = []
    snapshots   = []
    frame_iters = set(np.linspace(0, N_ITER-1, GIF_FRAMES, dtype=int))

    for it in range(N_ITER):
        eta   = ETA_0 / (1.0 + it * LR_DECAY)   # decaying step size
        grads = compute_gradients(hubs, data, active, asn)
        gnorm = np.linalg.norm(grads)
        scale = min(1.0, GRAD_CLIP/gnorm) if gnorm > 1e-9 else 1.0
        hubs  = np.clip(hubs - eta*grads*scale, 0.02, 0.98)
        asn   = get_assignments(data, hubs)
        if (it+1) % EDGE_EVERY == 0:
            active = prune_edges(hubs, active, asn)
        loss = compute_loss(hubs, data, active, asn)
        losses.append(loss)
        if it in frame_iters:
            snapshots.append((hubs.copy(), list(active), asn.copy(), loss, it+1))

    return data, snapshots, losses

# drawing
CLUSTER_COLORS = [
    "#f87171","#fbbf24","#34d399","#60a5fa",
    "#020202","#f472b6","#fb923c",
]

def draw_frame(ax, data, hubs, active, asn, loss, it):
    ax.clear()
    ax.set_facecolor(BG_COLOR)
    ax.set_xlim(-0.02, 1.02); ax.set_ylim(-0.02, 1.02)
    ax.set_aspect("equal"); ax.axis("off")
    n = len(hubs)

    # candidate edges (dim dashes)
    for i in range(n):
        for j in range(i+1, n):
            if np.linalg.norm(hubs[i]-hubs[j]) > CAND_THRESH: continue
            if canon((i,j)) not in [canon(e) for e in active]:
                ax.plot([hubs[i,0],hubs[j,0]], [hubs[i,1],hubs[j,1]],
                        color="#1f2a3e", lw=0.7, ls="--", zorder=1, alpha=0.5)

    # active edges
    for i,j in active:
        ax.plot([hubs[i,0],hubs[j,0]], [hubs[i,1],hubs[j,1]],
                color="#60a5fa", lw=2.2, alpha=0.85, zorder=2)

    # assignment lines
    for idx,pt in enumerate(data):
        h = hubs[asn[idx]]
        ax.plot([pt[0],h[0]], [pt[1],h[1]],
                color="#f87171", lw=0.35, alpha=0.12, zorder=2)

    # data points coloured by cluster
    for k in range(n):
        pts = data[asn==k]
        if len(pts):
            ax.scatter(pts[:,0], pts[:,1], s=14,
                       color=CLUSTER_COLORS[k % len(CLUSTER_COLORS)],
                       alpha=0.6, zorder=3, linewidths=0)

    # hubs
    wi    = np.array([np.sum(asn==i) for i in range(n)])
    sizes = 120 + wi*20
    ax.scatter(hubs[:,0], hubs[:,1], s=sizes, color="#6ee7b7",
               edgecolors="#07090d", linewidths=2, zorder=5)
    for i,(x,y) in enumerate(hubs):
        ax.text(x, y-0.055, f"x{i}", color="#6ee7b7",
                fontsize=7, ha="center", va="top",
                fontfamily="monospace", zorder=6)

    eta_now = ETA_0 / (1.0 + (it-1) * LR_DECAY)
    ax.set_title(
        f"Modified HP Filter  ·  iter {it}  ·  F = {loss:.1f}"
        f"  ·  |E| = {len(active)}  ·  η = {eta_now:.4f}",
        color=FG_COLOR, fontsize=9, fontfamily="monospace", pad=8, loc="left")

def add_loss_panel(ax2, losses, current_idx):
    ax2.clear()
    ax2.set_facecolor("#0c0f17")
    visible = losses[:current_idx]
    if visible:
        ax2.plot(range(1, len(visible)+1), visible, color="#6ee7b7", lw=1.3)
        ax2.fill_between(range(1, len(visible)+1), visible,
                         alpha=0.08, color="#6ee7b7")
        ymin, ymax = min(visible), max(visible)
        pad = max((ymax-ymin)*0.1, 1.0)
        ax2.set_ylim(ymin-pad, ymax+pad)
    ax2.set_xlim(1, N_ITER)
    ax2.tick_params(colors="#475569", labelsize=6)
    for sp in ax2.spines.values(): sp.set_edgecolor("#1f2a3e")
    ax2.set_facecolor("#0c0f17")
    ax2.set_xlabel("iter", color="#475569", fontsize=7)
    ax2.set_ylabel("F",    color="#475569", fontsize=7)
    ax2.set_title("Loss F(X,E)", color="#6ee7b7", fontsize=7, pad=4)

def save_static(data, snapshots, losses):
    hubs, active, asn, loss, it = snapshots[-1]
    fig = plt.figure(figsize=(11, 7), facecolor=BG_COLOR)
    ax1 = fig.add_axes([0.02, 0.02, 0.64, 0.88])   # main plot  (left 65%)
    ax2 = fig.add_axes([0.70, 0.10, 0.28, 0.30])   # loss panel (right, lower third)
    draw_frame(ax1, data, hubs, active, asn, loss, it)
    add_loss_panel(ax2, losses, len(losses))
    fig.savefig("hp_filter_final.png", dpi=150, facecolor=fig.get_facecolor())
    plt.close(fig)
    print("Saved hp_filter_final.png")

# Animated GIF
def save_gif(data, snapshots, losses):
    fig = plt.figure(figsize=(10, 6), facecolor=BG_COLOR)
    ax1 = fig.add_axes([0.02, 0.02, 0.64, 0.88])   # main plot  (left 65%)
    ax2 = fig.add_axes([0.70, 0.10, 0.28, 0.30])   # loss panel (right, lower third)
    frames = []

    for hubs, active, asn, loss, it in snapshots:
        draw_frame(ax1, data, hubs, active, asn, loss, it)
        add_loss_panel(ax2, losses, it)
        fig.canvas.draw()
        w, h = fig.canvas.get_width_height()
        buf = np.frombuffer(fig.canvas.tostring_rgb(), dtype=np.uint8).reshape(h,w,3)
        frames.append(Image.fromarray(buf))

    frames[0].save(
        "hp_filter_anim.gif",
        save_all=True, append_images=frames[1:],
        duration=int(1000/GIF_FPS), loop=0,
    )
    plt.close(fig)
    print("Saved hp_filter_anim.gif")


if __name__ == "__main__":
    print("Running optimisation...")
    data, snapshots, losses = run_optimization()
    print(f"Done. Final loss: {losses[-1]:.2f}  |  Frames: {len(snapshots)}")
    save_static(data, snapshots, losses)
    save_gif(data, snapshots, losses)
    print("All done!")

Running optimisation...
Done. Final loss: 249.97  |  Frames: 50
Saved hp_filter_final.png


C:\Users\emily\AppData\Local\Temp\ipykernel_21628\1898920194.py:348: MatplotlibDeprecationWarning: The tostring_rgb function was deprecated in Matplotlib 3.8 and will be removed in 3.10. Use buffer_rgba instead.
  buf = np.frombuffer(fig.canvas.tostring_rgb(), dtype=np.uint8).reshape(h,w,3)


Saved hp_filter_anim.gif
All done!
